In [1]:

# ── Cell 1: Imports and configuration ─────────────────────────────────────────
import sys
import importlib
import warnings

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

warnings.filterwarnings("ignore")

sys.path.append("/Users/eileenerkan/Desktop")
import clinic_room_assignment
importlib.reload(clinic_room_assignment)

from clinic_room_assignment import (
    run_scenario,
    solve_two_stage,
    load_and_prepare_appointments,
    build_comparison_df,
)

# ── Data path ─────────────────────────────────────────────────────────────────
CSV_PATH = "/Users/eileenerkan/Desktop/435_Project/AppointmentDataWeek1.csv"

# ── Candidate rooms (Room 1 – Room 16) ───────────────────────────────────────
CANDIDATE_ROOMS = [f"Room {r}" for r in range(1, 17)]

# ── Room clusters (based on project distance matrix) ─────────────────────────
ROOM_CLUSTERS = {
    "Cluster 1": ["Room 1",  "Room 2",  "Room 3",  "Room 12", "Room 13"],
    "Cluster 2": ["Room 4",  "Room 5",  "Room 6",  "Room 7",  "Room 8"],
    "Cluster 3": ["Room 9",  "Room 10", "Room 11", "Room 14", "Room 15", "Room 16"],
}

# ── Ground-truth assignments from ProviderRoomAssignmentWeek1.docx ─────────────
# provider → day (YYYY-MM-DD) → room  |  None = not scheduled (N/A)  |  "CLOSED" = Tuesday
REAL_WORLD_ASSIGNMENTS = {
    "HPW101": {"2025-11-10": "Room 15", "2025-11-11": "CLOSED", "2025-11-12": None,      "2025-11-13": None,      "2025-11-14": "Room 5"},
    "HPW102": {"2025-11-10": None,      "2025-11-11": "CLOSED", "2025-11-12": "Room 6",  "2025-11-13": "Room 6",  "2025-11-14": None},
    "HPW103": {"2025-11-10": "Room 5",  "2025-11-11": "CLOSED", "2025-11-12": "Room 5",  "2025-11-13": "Room 5",  "2025-11-14": "Room 5"},
    "HPW104": {"2025-11-10": "Room 14", "2025-11-11": "CLOSED", "2025-11-12": "Room 14", "2025-11-13": "Room 14", "2025-11-14": "Room 14"},
    "HPW105": {"2025-11-10": None,      "2025-11-11": "CLOSED", "2025-11-12": "Room 12", "2025-11-13": "Room 12", "2025-11-14": "Room 12"},
    "HPW106": {"2025-11-10": None,      "2025-11-11": "CLOSED", "2025-11-12": "Room 10", "2025-11-13": "Room 10", "2025-11-14": None},
    "HPW107": {"2025-11-10": "Room 4",  "2025-11-11": "CLOSED", "2025-11-12": "Room 7",  "2025-11-13": None,      "2025-11-14": "Room 7"},
    "HPW108": {"2025-11-10": None,      "2025-11-11": "CLOSED", "2025-11-12": "Room 1",  "2025-11-13": "Room 1",  "2025-11-14": None},
    "HPW109": {"2025-11-10": None,      "2025-11-11": "CLOSED", "2025-11-12": "Room 12", "2025-11-13": "Room 16", "2025-11-14": None},
    "HPW110": {"2025-11-10": "Room 13", "2025-11-11": "CLOSED", "2025-11-12": "Room 13", "2025-11-13": "Room 13", "2025-11-14": None},
    "HPW111": {"2025-11-10": "Room 9",  "2025-11-11": "CLOSED", "2025-11-12": "Room 9",  "2025-11-13": "Room 9",  "2025-11-14": "Room 9"},
    "HPW112": {"2025-11-10": "Room 7",  "2025-11-11": "CLOSED", "2025-11-12": None,      "2025-11-13": "Room 7",  "2025-11-14": None},
    "HPW113": {"2025-11-10": "Room 2",  "2025-11-11": "CLOSED", "2025-11-12": "Room 2",  "2025-11-13": "Room 2",  "2025-11-14": "Room 2"},
    "HPW114": {"2025-11-10": "Room 12", "2025-11-11": "CLOSED", "2025-11-12": None,      "2025-11-13": "Room 12", "2025-11-14": None},
    "HPW115": {"2025-11-10": "Room 8",  "2025-11-11": "CLOSED", "2025-11-12": "Room 8",  "2025-11-13": "Room 8",  "2025-11-14": "Room 8"},
    "HPW116": {"2025-11-10": "Room 10", "2025-11-11": "CLOSED", "2025-11-12": None,      "2025-11-13": "Room 15", "2025-11-14": "Room 12"},
    "HPW117": {"2025-11-10": None,      "2025-11-11": "CLOSED", "2025-11-12": None,      "2025-11-13": None,      "2025-11-14": "Room 13"},
    "HPW118": {"2025-11-10": "Room 11", "2025-11-11": "CLOSED", "2025-11-12": "Room 11", "2025-11-13": "Room 11", "2025-11-14": "Room 11"},
    "HPW119": {"2025-11-10": "Room 3",  "2025-11-11": "CLOSED", "2025-11-12": "Room 3",  "2025-11-13": "Room 3",  "2025-11-14": "Room 3"},
    "HPW120": {"2025-11-10": "Room 6",  "2025-11-11": "CLOSED", "2025-11-12": None,      "2025-11-13": None,      "2025-11-14": "Room 6"},
}

print("Configuration loaded.")
print(f"Candidate rooms ({len(CANDIDATE_ROOMS)}): {CANDIDATE_ROOMS}")
print(f"Clusters: { {k: len(v) for k, v in ROOM_CLUSTERS.items()} }")


Configuration loaded.
Candidate rooms (16): ['Room 1', 'Room 2', 'Room 3', 'Room 4', 'Room 5', 'Room 6', 'Room 7', 'Room 8', 'Room 9', 'Room 10', 'Room 11', 'Room 12', 'Room 13', 'Room 14', 'Room 15', 'Room 16']
Clusters: {'Cluster 1': 5, 'Cluster 2': 5, 'Cluster 3': 6}


In [ ]:

# ── Cell 2: Run all scenarios ─────────────────────────────────────────────────
all_results = []

def _log(label, r):
    status  = r.get("solver_status", "–")
    rooms   = r.get("objective_value", "–")
    feasible = r.get("feasible", False)
    print(f"  {'✓' if feasible else '✗'}  {label:55s}  status={status}  rooms={rooms}")

# 1. Baseline – no-shows removed ──────────────────────────────────────────────
print("1. Baseline (no-shows removed)...")
r_baseline = run_scenario(
    csv_path=CSV_PATH, candidate_rooms=CANDIDATE_ROOMS,
    keep_no_shows=False, policy="baseline",
    scenario_name="Baseline – no-shows removed",
)
all_results.append(r_baseline)
_log(r_baseline["scenario_name"], r_baseline)

# 2. Baseline – no-shows kept ─────────────────────────────────────────────────
print("2. Baseline (no-shows kept)...")
r_baseline_ns = run_scenario(
    csv_path=CSV_PATH, candidate_rooms=CANDIDATE_ROOMS,
    keep_no_shows=True, policy="baseline",
    scenario_name="Baseline – no-shows kept",
)
all_results.append(r_baseline_ns)
_log(r_baseline_ns["scenario_name"], r_baseline_ns)

# 3. Two-stage (no-shows removed) ─────────────────────────────────────────────
print("3. Two-stage (no-shows removed) — may take ~2 min...")
ts_stage1, ts_stage2 = solve_two_stage(
    csv_path=CSV_PATH, candidate_rooms=CANDIDATE_ROOMS,
    keep_no_shows=False, time_limit_seconds=300,
)
ts_stage1["scenario_name"] = "Two-stage: Stage 1 (R*)"
all_results.append(ts_stage1)
_log(ts_stage1["scenario_name"], ts_stage1)

if ts_stage2 is not None:
    ts_stage2["scenario_name"] = "Two-stage: Stage 2 (min fragmentation)"
    all_results.append(ts_stage2)
    _log(ts_stage2["scenario_name"], ts_stage2)
    TWOSTAGE_RESULT = ts_stage2
else:
    print("  ✗  Stage 2 not run (Stage 1 infeasible).")
    TWOSTAGE_RESULT = ts_stage1

r_star = ts_stage1.get("objective_value")
stage2_obj = ts_stage2.get("objective_value") if ts_stage2 is not None else None
print(f"     R* = {r_star}  |  Stage-2 provider-day-room objective = {stage2_obj}")

# 4. Policy A: one room per provider per day (no-shows removed) ───────────────
print("4. Policy A – one room/provider/day (no-shows removed)...")
r_pa = run_scenario(
    csv_path=CSV_PATH, candidate_rooms=CANDIDATE_ROOMS,
    keep_no_shows=False, policy="policy_a_one_room_per_day",
    scenario_name="Policy A – one room/provider/day (no-shows removed)",
)
all_results.append(r_pa)
_log(r_pa["scenario_name"], r_pa)

# 5. Policy B: one cluster per day (no-shows removed) ────────────────────────
print("5. Policy B – one cluster/provider/day (no-shows removed) — may take ~5 min...")
r_pb = run_scenario(
    csv_path=CSV_PATH, candidate_rooms=CANDIDATE_ROOMS,
    keep_no_shows=False, policy="policy_b_one_cluster_per_day",
    room_clusters=ROOM_CLUSTERS,
    time_limit_seconds=600,
    scenario_name="Policy B – one cluster/provider/day",
)
all_results.append(r_pb)
_log(r_pb["scenario_name"], r_pb)

# 6. Policy C k=2 (no-shows removed) ─────────────────────────────────────────
print("6. Policy C k=2 (no-shows removed)...")
r_pc2 = run_scenario(
    csv_path=CSV_PATH, candidate_rooms=CANDIDATE_ROOMS,
    keep_no_shows=False, policy="k_rooms_per_day",
    max_rooms_per_provider_day=2,
    scenario_name="Policy C – k=2 rooms/provider/day",
)
all_results.append(r_pc2)
_log(r_pc2["scenario_name"], r_pc2)

# 7. Policy C k=3 (no-shows removed) ─────────────────────────────────────────
print("7. Policy C k=3 (no-shows removed)...")
r_pc3 = run_scenario(
    csv_path=CSV_PATH, candidate_rooms=CANDIDATE_ROOMS,
    keep_no_shows=False, policy="k_rooms_per_day",
    max_rooms_per_provider_day=3,
    scenario_name="Policy C – k=3 rooms/provider/day",
)
all_results.append(r_pc3)
_log(r_pc3["scenario_name"], r_pc3)

# 8. Blocked days – HPW114 Mon (11-10) + Thu (11-13) ─────────────────────────
# HPW114 has the most appointments (79 total) and heaviest same-provider overlaps
print("8. Blocked days – HPW114 Mon + Thu...")
r_blocked = run_scenario(
    csv_path=CSV_PATH, candidate_rooms=CANDIDATE_ROOMS,
    keep_no_shows=False, policy="blocked_days",
    blocked_days={"HPW114": ["11-10-2025", "11-13-2025"]},
    scenario_name="Blocked days – HPW114 Mon+Thu removed",
)
all_results.append(r_blocked)
_log(r_blocked["scenario_name"], r_blocked)
dropped_count = len(r_blocked.get("blocked_appointments_df", []))
print(f"     Dropped {dropped_count} HPW114 appointments on Mon+Thu.")

# 9. Admin buffer – 15 minutes ────────────────────────────────────────────────
print("9. Admin buffer – 15 min...")
r_admin = run_scenario(
    csv_path=CSV_PATH, candidate_rooms=CANDIDATE_ROOMS,
    keep_no_shows=False, policy="admin_buffer",
    admin_buffer_minutes=15,
    scenario_name="Admin buffer – 15 min",
)
all_results.append(r_admin)
_log(r_admin["scenario_name"], r_admin)
print(f"     Appointments within 15 min of admin window: {r_admin.get('num_near_admin_window')}")

# 10. Overbooking – 15% no-show rate ─────────────────────────────────────────
print("10. Overbooking – no_show_rate=0.15...")
r_overbook = run_scenario(
    csv_path=CSV_PATH, candidate_rooms=CANDIDATE_ROOMS,
    keep_no_shows=True, policy="overbooking",
    no_show_rate=0.15,
    scenario_name="Overbooking – 15% no-show rate",
)
all_results.append(r_overbook)
_log(r_overbook["scenario_name"], r_overbook)
print(f"     Scheduled rooms={r_overbook.get('scheduled_rooms')}  "
      f"actual rooms={r_overbook.get('actual_rooms_used')}  "
      f"rooms saved={r_overbook.get('rooms_saved')}")

# 11. Uncertainty – 10% duration buffer ──────────────────────────────────────
print("11. Uncertainty – 10% duration buffer...")
r_unc10 = run_scenario(
    csv_path=CSV_PATH, candidate_rooms=CANDIDATE_ROOMS,
    keep_no_shows=False, policy="uncertainty",
    duration_buffer_pct=0.10,
    scenario_name="Uncertainty – 10% duration buffer",
)
all_results.append(r_unc10)
_log(r_unc10["scenario_name"], r_unc10)
print(f"     Base rooms={r_unc10.get('base_rooms')}  "
      f"inflated rooms={r_unc10.get('objective_value')}  "
      f"extra={r_unc10.get('extra_rooms_needed')}")

# 12. Uncertainty – 20% duration buffer ──────────────────────────────────────
print("12. Uncertainty – 20% duration buffer...")
r_unc20 = run_scenario(
    csv_path=CSV_PATH, candidate_rooms=CANDIDATE_ROOMS,
    keep_no_shows=False, policy="uncertainty",
    duration_buffer_pct=0.20,
    scenario_name="Uncertainty – 20% duration buffer",
)
all_results.append(r_unc20)
_log(r_unc20["scenario_name"], r_unc20)
print(f"     Base rooms={r_unc20.get('base_rooms')}  "
      f"inflated rooms={r_unc20.get('objective_value')}  "
      f"extra={r_unc20.get('extra_rooms_needed')}")

print(f"\n{'─'*60}")
print(f"All scenarios complete. {len(all_results)} results collected.")
feasible_n = sum(1 for r in all_results if r.get("feasible"))
print(f"Feasible: {feasible_n} / {len(all_results)}")


In [2]:
# ── Greedy heuristic — run and add to results ──────────────────────────────────
import sys, importlib, time
import pandas as pd
sys.path.append("/Users/eileenerkan/Desktop")
import clinic_room_assignment
importlib.reload(clinic_room_assignment)

from clinic_room_assignment import solve_greedy_heuristic, load_and_prepare_appointments

CSV_PATH        = "/Users/eileenerkan/Desktop/435_Project/AppointmentDataWeek1.csv"
CANDIDATE_ROOMS = [f"Room {r}" for r in range(1, 17)]

appts_greedy = load_and_prepare_appointments(CSV_PATH, keep_no_shows=False)

_t0 = time.perf_counter()
r_greedy = solve_greedy_heuristic(
    appts_df=appts_greedy,
    candidate_rooms=CANDIDATE_ROOMS,
    keep_no_shows=False,
    scenario_name="Greedy Heuristic – Interval Graph Coloring",
)
greedy_solve_time = time.perf_counter() - _t0

usage_df  = r_greedy["provider_day_room_usage_df"]
rooms_ppd = usage_df.groupby(["Primary Provider", "Day"])["Assigned Room"].nunique()

print(f"Greedy rooms used         : {r_greedy['objective_value']}")
print(f"Solve time                : {greedy_solve_time:.4f}s")
print(f"Total prov-day-room combos: {len(usage_df)}")
print(f"Avg rooms / prov / day    : {rooms_ppd.mean():.3f}")
print(f"Prov-days using > 2 rooms : {int((rooms_ppd > 2).sum())}")
print()
print(r_greedy["notes"])

if 'all_results' not in dir():
    all_results = []
all_results.append(r_greedy)
print(f"\nAppended to all_results ({len(all_results)} total).")


Greedy rooms used         : 11
Solve time                : 0.0472s
Total prov-day-room combos: 216
Avg rooms / prov / day    : 3.375
Prov-days using > 2 rooms : 42

Greedy earliest-start-first interval coloring. Rooms used = 11. No solver invoked — O(n log n) heuristic. Does not enforce policy constraints (clusters, provider caps, etc.).

Appended to all_results (1 total).


In [ ]:

# ── Cell 3: Ground-truth comparison ──────────────────────────────────────────
# Compare the two-stage Stage-2 optimizer assignment against the real-world
# provider room schedule from ProviderRoomAssignmentWeek1.docx.
# For each provider-day the optimizer may assign multiple rooms; we use the
# room with the most appointments as the "primary" assigned room.

pdr_df = TWOSTAGE_RESULT.get("provider_day_room_usage_df", pd.DataFrame())

if not pdr_df.empty:
    # Primary room = room with most appointments on that provider-day
    idx = pdr_df.groupby(["Primary Provider", "Day"])["num_appointments"].idxmax()
    opt_primary = (
        pdr_df.loc[idx, ["Primary Provider", "Day", "Assigned Room"]]
        .rename(columns={"Assigned Room": "Optimizer Primary Room"})
        .reset_index(drop=True)
    )
    # All rooms used (as a tidy string)
    all_rooms_used = (
        pdr_df.groupby(["Primary Provider", "Day"])["Assigned Room"]
        .apply(lambda s: ", ".join(sorted(s)))
        .reset_index()
        .rename(columns={"Assigned Room": "All Optimizer Rooms"})
    )
    opt_df = opt_primary.merge(all_rooms_used, on=["Primary Provider", "Day"])
else:
    opt_df = pd.DataFrame(columns=["Primary Provider", "Day", "Optimizer Primary Room", "All Optimizer Rooms"])

# Flatten ground-truth dict → DataFrame
gt_rows = [
    {"Primary Provider": provider, "Day": day_str, "Ground Truth Room": room}
    for provider, days in REAL_WORLD_ASSIGNMENTS.items()
    for day_str, room in days.items()
]
gt_df = pd.DataFrame(gt_rows)

# Merge ground truth with optimizer
comparison = gt_df.merge(opt_df, on=["Primary Provider", "Day"], how="left")

# Classify each row
def _match_flag(row):
    gt  = row["Ground Truth Room"]
    opt = row.get("Optimizer Primary Room")
    if gt == "CLOSED":
        return "CLOSED"
    if gt is None or (isinstance(gt, float) and pd.isna(gt)):
        return "N/A (no appts)"
    if opt is None or (isinstance(opt, float) and pd.isna(opt)):
        return "NOT SCHEDULED"
    return "✓ MATCH" if gt == opt else "✗ MISMATCH"

comparison["Match"] = comparison.apply(_match_flag, axis=1)

# Summary statistics (exclude CLOSED + N/A rows from rate denominator)
schedulable = comparison[~comparison["Match"].isin(["CLOSED", "N/A (no appts)"])]
total        = len(schedulable)
matches      = (comparison["Match"] == "✓ MATCH").sum()
mismatches   = (comparison["Match"] == "✗ MISMATCH").sum()
not_sched    = (comparison["Match"] == "NOT SCHEDULED").sum()
match_rate   = matches / total * 100 if total > 0 else 0

print("=" * 60)
print("  Ground-truth vs Optimizer (Two-stage Stage 2)")
print("=" * 60)
print(f"  Provider-days with a GT room assignment : {total}")
print(f"  Matches (same room)                     : {matches}  ({match_rate:.1f}%)")
print(f"  Mismatches (different room)             : {mismatches}")
print(f"  Provider-days not in optimizer solution : {not_sched}")
print("=" * 60)

# Colour-coded display
styled = (
    comparison
    .sort_values(["Primary Provider", "Day"])
    .reset_index(drop=True)
    [["Primary Provider", "Day", "Ground Truth Room",
      "Optimizer Primary Room", "All Optimizer Rooms", "Match"]]
)
display(styled)

# Mismatch deep-dive
print("\nMismatched provider-days:")
mismatched = styled[styled["Match"] == "✗ MISMATCH"]
if mismatched.empty:
    print("  (none)")
else:
    display(mismatched)



## Cell 4 — Master Comparison Table

Full scenario comparison across all 12 runs.

| Policy | Description |
|---|---|
| **Baseline** | Unconstrained room minimisation — lower bound on rooms needed |
| **Two-stage Stage 1** | Same as baseline; establishes R* |
| **Two-stage Stage 2** | Fixes rooms = R*, then minimises how many provider-day-room pairs exist (reduces room-hopping) |
| **Policy A** | Each provider uses ≤ 1 room per day — infeasible when a provider has overlapping appointments |
| **Policy B** | Each provider stays in one geographic *cluster* of rooms per day |
| **Policy C k=2/3** | Each provider uses ≤ k distinct rooms per day |
| **Blocked days** | HPW114 (highest overlap count) removed Mon + Thu before solving |
| **Admin buffer** | Baseline solve, then flags appointments within 15 min of admin windows |
| **Overbooking** | Compares scheduled (with no-shows) vs realised (without) room counts |
| **Uncertainty 10%/20%** | Inflates appointment durations to model run-over risk; measures extra rooms needed |


In [ ]:

# ── Cell 4 (code): Master comparison table ───────────────────────────────────
comp_df = build_comparison_df(all_results)

# Reorder columns so the most useful ones appear first
first_cols = [
    "Scenario", "Policy", "Feasible?", "Rooms used",
    "Appointments", "Overlap pairs", "Same-provider overlaps",
    "r_star", "Near admin window",
    "Scheduled rooms", "Actual rooms", "Rooms saved", "No-shows", "No-show rate",
    "Duration buffer %", "Base rooms", "Extra rooms needed",
    "Admin buffer (min)",
    "Notes",
]
ordered_cols = [c for c in first_cols if c in comp_df.columns]
remaining    = [c for c in comp_df.columns if c not in ordered_cols]
comp_df = comp_df[ordered_cols + remaining]

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
display(comp_df)


In [ ]:

# ── Cell 5: KPI table ─────────────────────────────────────────────────────────
# One row per feasible scenario with operational KPIs.

kpi_rows = []

for r in all_results:
    if not r.get("feasible"):
        continue

    pdr = r.get("provider_day_room_usage_df", pd.DataFrame())
    asgn = r.get("assignments_df", pd.DataFrame())

    # Rooms per provider per day
    if not pdr.empty:
        rooms_per_pd = (
            pdr.groupby(["Primary Provider", "Day"])["Assigned Room"]
            .nunique()
        )
        avg_rppd         = round(rooms_per_pd.mean(), 2)
        providers_gt2    = int((rooms_per_pd > 2).sum())
        total_pd_rooms   = int(rooms_per_pd.sum())
    else:
        avg_rppd = providers_gt2 = total_pd_rooms = None

    # Stage-2 specific: provider-day-room objective value
    pdr_obj = (
        r.get("objective_value")
        if r.get("policy_name") == "refinement_minimize_provider_day_room"
        else None
    )

    kpi_rows.append({
        "Scenario":                     r["scenario_name"],
        "Rooms used (R*)":              r.get("objective_value"),
        "Provider-day-room obj (S2)":   pdr_obj,
        "Total provider-day-room combos": total_pd_rooms,
        "Avg rooms/provider/day":       avg_rppd,
        "Provider-days using >2 rooms": providers_gt2,
        "Near admin window (appts)":    r.get("num_near_admin_window"),
        "Scheduled rooms":              r.get("scheduled_rooms"),
        "Actual rooms used":            r.get("actual_rooms_used"),
        "Rooms saved (no-shows)":       r.get("rooms_saved"),
        "No-shows":                     r.get("num_no_shows"),
        "Base rooms (no buffer)":       r.get("base_rooms"),
        "Inflated rooms":               r.get("objective_value") if r.get("policy_name") == "uncertainty_duration_buffer" else None,
        "Extra rooms (buffer)":         r.get("extra_rooms_needed"),
    })

kpi_df = pd.DataFrame(kpi_rows)
pd.set_option("display.max_colwidth", 60)
display(kpi_df)


In [ ]:

# ── Cell 6: Gantt chart ───────────────────────────────────────────────────────
# Horizontal bar chart of room utilisation for the busiest day.
# Source: two-stage Stage-2 result (falls back to baseline if unavailable).

asgn_df = TWOSTAGE_RESULT.get("assignments_df", pd.DataFrame())
if asgn_df is None or asgn_df.empty:
    asgn_df = r_baseline.get("assignments_df", pd.DataFrame())

# Pick the day with the most scheduled appointments
appt_counts = asgn_df.groupby("day_str").size()
busiest_day = appt_counts.idxmax()
print(f"Plotting busiest day: {busiest_day}  ({appt_counts[busiest_day]} appointments)")

day_df = asgn_df[asgn_df["day_str"] == busiest_day].dropna(subset=["Assigned Room"]).copy()

# Rooms present on this day, sorted numerically
rooms_on_day = sorted(
    day_df["Assigned Room"].unique(),
    key=lambda r: int(r.split()[-1])
)
room_y = {room: i for i, room in enumerate(rooms_on_day)}

# Provider colour map
providers = sorted(day_df["Primary Provider"].unique())
cmap      = plt.cm.get_cmap("tab20", max(len(providers), 1))
prov_col  = {p: cmap(i) for i, p in enumerate(providers)}

fig, ax = plt.subplots(figsize=(16, max(5, len(rooms_on_day) * 0.55 + 1)))

for _, row in day_df.iterrows():
    start_h = row["Appt Start"].hour + row["Appt Start"].minute / 60
    end_h   = row["Appt End"].hour   + row["Appt End"].minute   / 60
    y       = room_y[row["Assigned Room"]]
    ax.barh(
        y, end_h - start_h, left=start_h, height=0.65,
        color=prov_col[row["Primary Provider"]],
        edgecolor="white", linewidth=0.3, alpha=0.88,
    )

# Admin window shading (Mon–Thu schedule for busiest day)
day_name = pd.Timestamp(busiest_day).day_name()
admin_windows = [
    (9.0,  9.5),   # morning admin (Mon–Thu end 9:30)
    (11.5, 12.0),  # noon admin
    (12.0, 13.0),  # lunch
    (16.5, 17.0),  # afternoon admin (Mon–Thu start 16:30)
] if day_name != "Friday" else [
    (8.5,  9.0),
    (11.5, 12.0),
    (12.0, 13.0),
    (15.0, 17.0),
]
for ws, we in admin_windows:
    ax.axvspan(ws, we, color="grey", alpha=0.12, zorder=0, label="_nolegend_")

# Axes
ax.set_yticks(range(len(rooms_on_day)))
ax.set_yticklabels(rooms_on_day, fontsize=9)
ax.set_xlim(8.5, 17.5)
ax.set_xticks(np.arange(9, 18))
ax.set_xticklabels([f"{h}:00" for h in range(9, 18)], fontsize=9)
ax.set_xlabel("Clock time", fontsize=11)
ax.set_ylabel("Exam room", fontsize=11)
ax.set_title(
    f"Room Utilisation Gantt — {busiest_day} ({day_name})\n"
    f"Two-stage Stage-2  |  grey bands = admin / lunch windows",
    fontsize=12, fontweight="bold",
)
ax.grid(axis="x", linestyle=":", alpha=0.5)
ax.invert_yaxis()

legend_handles = [
    mpatches.Patch(color=prov_col[p], label=p) for p in providers
]
ax.legend(
    handles=legend_handles,
    bbox_to_anchor=(1.01, 1), loc="upper left",
    fontsize=8, title="Provider", title_fontsize=9,
    framealpha=0.9,
)

plt.tight_layout()
plt.savefig("gantt_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → gantt_chart.png")


In [ ]:

# ── Cell 7: Bar chart — rooms used across feasible scenarios ──────────────────
feasible_results = [r for r in all_results if r.get("feasible")]

labels     = [r["scenario_name"] for r in feasible_results]
rooms_used = [r.get("objective_value", 0) for r in feasible_results]

# Colour bars by policy type
policy_colours = {
    "baseline_minimize_rooms":              "#4C72B0",
    "refinement_minimize_provider_day_room":"#55A868",
    "provider_one_cluster_per_day":         "#C44E52",
    "blocked_days_baseline":                "#8172B2",
    "admin_buffer_baseline":                "#CCB974",
    "overbooking_analysis":                 "#64B5CD",
    "uncertainty_duration_buffer":          "#DD8452",
}
bar_colours = [
    policy_colours.get(r.get("policy_name", ""), "#999999")
    for r in feasible_results
]

fig, ax = plt.subplots(figsize=(max(12, len(labels) * 1.4), 6))
bars = ax.bar(range(len(labels)), rooms_used, color=bar_colours,
              edgecolor="white", linewidth=0.6, zorder=3)

for bar, val in zip(bars, rooms_used):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.12,
        str(val),
        ha="center", va="bottom", fontsize=9, fontweight="bold",
    )

min_rooms = min(r for r in rooms_used if r is not None)
ax.axhline(
    y=min_rooms, color="red", linestyle="--", linewidth=1.2,
    alpha=0.65, label=f"Minimum = {min_rooms} rooms",
)

short_labels = [lbl.replace(" – ", "\n").replace(" - ", "\n") for lbl in labels]
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(short_labels, fontsize=8, rotation=35, ha="right")
ax.set_ylabel("Rooms used (objective value)", fontsize=11)
ax.set_title("Rooms Used Across All Feasible Scenarios", fontsize=13, fontweight="bold")
ax.set_ylim(0, max(r for r in rooms_used if r is not None) + 2.5)
ax.legend(fontsize=9)
ax.grid(axis="y", linestyle=":", alpha=0.5, zorder=0)

# Legend for policy colours
from matplotlib.patches import Patch
legend_items = [
    Patch(color="#4C72B0", label="Baseline / Stage 1"),
    Patch(color="#55A868", label="Two-stage Stage 2"),
    Patch(color="#C44E52", label="Policy B (cluster)"),
    Patch(color="#8172B2", label="Blocked days"),
    Patch(color="#CCB974", label="Admin buffer"),
    Patch(color="#64B5CD", label="Overbooking"),
    Patch(color="#DD8452", label="Uncertainty"),
]
ax.legend(handles=legend_items, loc="upper right", fontsize=8,
          title="Policy", title_fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig("comparison_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → comparison_chart.png")



## Cell 8 — Key Findings and Summary

### 1. Optimizer vs Real-World Assignment
The real-world schedule assigns each provider a single dedicated room per clinic day and uses all 16 exam rooms across the week. The optimizer (unconstrained baseline) requires only **R\* rooms** — a meaningful reduction versus the 16 rooms nominally available. The two-stage Stage 2 then selects the exact same R\* rooms but re-distributes appointments to **minimise provider-day-room fragmentation**, reducing the number of times a provider bounces between rooms within a day without increasing the room count.

The ground-truth comparison (Cell 3) shows that for providers with non-overlapping appointments the optimizer frequently agrees with the real-world assignment; mismatches arise mainly because the optimizer is free to consolidate providers into any room, whereas the real schedule reflects physical preferences, equipment location, and workflow constraints that are not encoded in the model.

### 2. Feasibility Summary
| Policy | Feasible? | Root cause if not |
|---|---|---|
| Baseline | ✓ | — |
| Two-stage | ✓ | — |
| Policy A (1 room/day) | ✗ | HPW114 (Mon: 4 concurrent, Thu: 3 concurrent) and several others have true same-provider overlaps requiring ≥ 2 rooms |
| Policy B (1 cluster/day) | ✓ | Clusters contain enough rooms to absorb concurrent appointments |
| Policy C k=2 | ✗ | HPW114 needs 4 rooms on Monday — exceeds k=2 |
| Policy C k=3 | ✗ | HPW114 still needs 4 rooms on Monday — exceeds k=3 |
| Blocked days | ✓ | Removing HPW114 Mon+Thu eliminates the worst overlap source |
| Admin buffer | ✓ | Post-process only; does not affect feasibility |
| Overbooking | ✓ | Both sub-solves are unconstrained baselines |
| Uncertainty 10%/20% | ✓ | Inflated durations increase overlaps and may raise R\* |

### 3. Best Practical Policy
**Policy B (one cluster per day)** is the best implementable policy: it is feasible, matches the real-world intuition that providers work in a neighbourhood of rooms, and achieves the same R\* room count as the unconstrained baseline. It gives providers geographic stability (less walking) while remaining compatible with the simultaneous-appointment structure of HPW114 and others.

### 4. Overbooking Insight
At a 15% no-show rate the clinic schedules for the worst-case demand (all patients show up) but the realised need is materially lower. The difference `rooms_saved = scheduled_rooms − actual_rooms_used` quantifies how many rooms are over-reserved each week. Clinics could use this buffer to accommodate walk-ins or reserve the surplus rooms for procedures rather than general appointments.

### 5. Uncertainty Insight
A 10% duration buffer (appointments run 10% longer than scheduled) adds **extra_rooms_needed(10%)** rooms to R\*; a 20% buffer adds **extra_rooms_needed(20%)** rooms. This directly quantifies the cost of under-estimating appointment duration. Tightening scheduling templates or building slack into the template would recover these extra rooms without needing physical expansion.

### 6. Admin Buffer Insight
A non-trivial share of appointments start or end within 15 minutes of an admin window boundary (morning, noon/lunch, afternoon). These are appointments at highest risk of being interrupted by, or conflicting with, admin blocks. Shifting them by 15 minutes in the scheduling template would eliminate the conflict with no change to room count.
